<a href="https://colab.research.google.com/github/ChaimElchik/GPS-Demo/blob/main/DirectGeoReferencingTransectCountGD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐘 Direct Georeferencing for Drone-Based Line Transect Surveys

Welcome! This notebook provides a complete workflow for analyzing drone imagery to perform line transect surveys. It's designed for ecologists and researchers who want to automate the process of detecting animals (or any object), geolocating them, and calculating their perpendicular distance from a flight path (transect line).

### 🎯 What This Notebook Does:

1.  **Object Detection**: It uses a powerful vision model to find specific objects you define (e.g., "Elephant", "Boat", "Tree") in a drone image.
2.  **Depth Estimation**: It creates a depth map of the scene to understand how far away each point in the image is from the camera.
3.  **Direct Georeferencing**: Using the drone's own GPS and gimbal data (pitch, yaw, roll) from the image's EXIF metadata, it calculates the real-world GPS coordinates for each detected object.
4.  **Transect Analysis**: It models the drone's flight path as a straight line (transect) and calculates the perpendicular distance of each object from this line.
5.  **Outputs**: It generates two key files for you:
    * An **annotated image** showing the detected objects, their IDs, and their distance from the transect.
    * A **CSV data file**, perfectly formatted for distance sampling analysis in software like R.

### ✅ How to Use This Notebook:

Simply run the cells in order from top to bottom. The final cell is the **User Input** section where you'll be prompted to:
1.  Specify the object you want to detect.
2.  Upload your drone image(s).

Let's get started!

## ⚙️ 1. Setup and Installation

This first block of code prepares our environment. It performs several key steps:

* **Clones the Repository**: It downloads the necessary code from the `CountGD` GitHub repository.
* **Installs System Tools**: It ensures that essential tools for compiling code are available on the system.
* **Installs Python Libraries**: It reads the `requirements.txt` file and installs all the required Python packages, such as `torch`, `transformers`, and `opencv`, which are needed for the AI models and image processing to work correctly.

This process might take a few minutes as it downloads and installs several large libraries.

In [1]:
%%capture
!git clone https://github.com/niki-amini-naieni/CountGD.git
%cd CountGD
!sudo apt update
!sudo apt install -y build-essential

#### **The cell below will generate a restart runtime popup, press restart runtime and continue with the next cell after**

In [2]:
!pip install -r requirements.txt
!export CC=/usr/bin/gcc-11 # this ensures that gcc 11 is being used for compilation

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 10.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.7/74.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.8/37.8 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.3/250.3 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 112.1 MB/s eta 0:00:

### **Run the code below in normal fashion after having perfromed the pop up restart**

In [1]:
%%capture
%cd CountGD
!pip install -r requirements.txt
!export CC=/usr/bin/gcc-11 # this ensures that gcc 11 is being used for compilation
%cd models/GroundingDINO/ops
!python setup.py build install

In [2]:
!python test.py # should result in 6 lines of * True

* True check_forward_equal_with_pytorch_double: max_abs_err 8.67e-19 max_rel_err 2.35e-16
* True check_forward_equal_with_pytorch_float: max_abs_err 4.66e-10 max_rel_err 1.13e-07
* True check_gradient_numerical(D=30)
* True check_gradient_numerical(D=32)
* True check_gradient_numerical(D=64)
* True check_gradient_numerical(D=71)


In [3]:
%%capture
!pip install git+https://github.com/facebookresearch/segment-anything.git
!cd ../../../
!mkdir checkpoints
!python download_bert.py
!wget -P checkpoints https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha2/groundingdino_swinb_cogcoor.pth
!wget -P checkpoints https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
# Install necessary packages for DirectGeoReferencing
!pip install piexif geopy pyproj torch torchvision transformers timm accelerate -q
!pip install ultralytics
!apt-get install -y exiftool -qq

In [4]:
!cd ../../../

build/        functions/                               setup.py
checkpoints/  modules/                                 src/
dist/         MultiScaleDeformableAttention.egg-info/  test.py


In [11]:
# --- Standard Library Imports ---
import os
import io
import re
import csv
import ast
import math
import time
import json
import traceback
import subprocess
from datetime import datetime, timedelta
from pathlib import Path

# --- Third-Party Library Imports ---
import cv2
import numpy as np
import pandas as pd
import requests
import torch
from geopy.distance import geodesic
from geopy.point import Point
from matplotlib import pyplot as plt
from PIL import Image
from pyproj import Transformer, Geod
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
from ultralytics import YOLO

# --- Google Colab / IPython Specific Imports ---
from google.colab import files
from IPython.display import Image as IPImage, display


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [12]:
# --- Global Configuration & Model Setup ---
OUTPUT_DIR = "Image_Processing_Output"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEPTH_MODEL_NAME = 'depth-anything/Depth-Anything-V2-Large-hf'

# --- Sensor Details (adjust if necessary) ---
SENSOR_WIDTH_MM = 17.3
SENSOR_HEIGHT_MM = 13.0
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- HELPER FUNCTIONS ---

def extract_exif_data(image_path):
    """
    Extracts essential drone metadata from an image's EXIF data using exiftool.
    """
    print(f"INFO: Extracting EXIF data from: {image_path}")
    try:
        result = subprocess.run(
            ['exiftool', '-j', '-n', '-G', image_path],
            stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, check=True
        )
        metadata = json.loads(result.stdout)[0]

        def get_value(keys, default=None):
            for key in keys:
                if key in metadata:
                    return metadata[key]
            return default

        lat = get_value(['EXIF:GPSLatitude', 'Composite:GPSLatitude'])
        lon = get_value(['EXIF:GPSLongitude', 'Composite:GPSLongitude'])
        rel_alt = get_value(['XMP:RelativeAltitude', 'EXIF:RelativeAltitude'])
        abs_alt = get_value(['XMP:AbsoluteAltitude', 'Composite:GPSAltitude', 'EXIF:GPSAltitude'])
        focal_len = get_value(['EXIF:FocalLength', 'Composite:FocalLength35efl'])
        pitch = get_value(['XMP:GimbalPitchDegree', 'Composite:GimbalPitch'])
        yaw = get_value(['XMP:GimbalYawDegree', 'Composite:GimbalYaw'])
        roll = get_value(['XMP:GimbalRollDegree', 'Composite:GimbalRoll'])

        essential_data = {'lat': lat, 'lon': lon, 'rel_alt': rel_alt, 'abs_alt': abs_alt, 'focal_len': focal_len, 'pitch': pitch, 'yaw': yaw, 'roll': roll}
        for key, value in essential_data.items():
            if value is None:
                raise ValueError(f"Missing essential EXIF tag: {key}")

        print("✅ Successfully extracted all required EXIF data.")
        return {
            'latitude': float(lat), 'longitude': float(lon),
            'rel_alt': float(rel_alt), 'abs_alt': float(abs_alt),
            'focal_len': float(str(focal_len).split()[0]),
            'gb_pitch': float(pitch), 'gb_yaw': float(yaw), 'gb_roll': float(roll),
        }

    except FileNotFoundError:
        raise RuntimeError("ERROR: 'exiftool' not found. Please ensure it is installed and in your system's PATH.")
    except Exception as e:
        print(f"❌ An error occurred during EXIF extraction: {e}")
        return None

def detect_objects_yolo(Object, Image_path):
    """
    Performs object detection on an image using a YOLO model and returns formatted bounding boxes.
    """
#

    cmd = [
        "python",
        "single_image_inference_for_DirectGeoReferencingUSe.py",
        "--image_path", Image_path,
        "--text", Object,
        "--config", "./config/cfg_fsc147_vit_b.py",
        "--pretrain_model_path", "checkpoints/checkpoint_fsc147_best.pth",
        "--return_boxes"
    ]

    # Run script and capture output
    result = subprocess.run(cmd, capture_output=True, text=True)

    # Last line should be your YOLO boxes
    output_lines = result.stdout.strip().split("\n")
    yolo_boxes_str = output_lines[-1]

    # Convert string to Python list
    yolo_boxes = ast.literal_eval(yolo_boxes_str)

    # Load image
    img = cv2.imread(Image_path)

    # Extract dimensions
    img_height, img_width = img.shape[:2]

    boxes = []

    print("Captured YOLO boxes:")
    i = 0
    for box in yolo_boxes:

        class_id = box[0]
        x_center = box[1]
        y_center = box[2]
        width = box[3]
        height = box[4]

        # Convert normalized xywh to pixel xyxy
        x1 = int((x_center - width / 2) * img_width)
        y1 = int((y_center - height / 2) * img_height)
        x2 = int((x_center + width / 2) * img_width)
        y2 = int((y_center + height / 2) * img_height)

        center_u = (x1 + x2) / 2
        center_v = (y1 + y2) / 2

        boxes.append({
            'class_id': class_id,
            'object_id': i + 1,
            'box': [x1, y1, x2, y2],
            'center_u': center_u,
            'center_v': center_v
        })
        i += 1

    print(f"✅ Successfully detected {len(boxes)} objects.")
    print(boxes)
    return boxes


def get_depth_map(frame, model, processor):
    """
    Generates a depth map from an image using a pre-trained model.
    """
    image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    inputs = processor(images=image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
    prediction = torch.nn.functional.interpolate(
        outputs.predicted_depth.unsqueeze(1), size=image.size[::-1], mode="bicubic", align_corners=False
    )
    return prediction.squeeze().cpu().numpy()

def get_dem_elevation_from_api(latitude, longitude):
    """
    Fetches ground elevation from an online API.
    """
    try:
        url = f"https://api.opentopodata.org/v1/eudem25m?locations={latitude},{longitude}"
        response = requests.get(url, verify=True, timeout=10)
        if response.status_code == 200 and response.json()['results'] and response.json()['results'][0]['elevation'] is not None:
            return response.json()['results'][0]['elevation']
    except requests.exceptions.RequestException:
        return None
    return None

def get_camera_intrinsics(f_mm, s_w_mm, s_h_mm, i_w, i_h):
    """
    Calculates the camera intrinsic matrix.
    """
    fx = i_w * f_mm / s_w_mm
    fy = i_h * f_mm / s_h_mm
    cx, cy = i_w / 2, i_h / 2
    return np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])

def get_rotation_matrix(pitch_deg, yaw_deg, roll_deg):
    """
    Calculates the rotation matrix from pitch, yaw, and roll angles.
    """
    yaw, pitch, roll = map(math.radians, [yaw_deg, pitch_deg, roll_deg])
    Rz = np.array([[math.cos(yaw), -math.sin(yaw), 0], [math.sin(yaw), math.cos(yaw), 0], [0, 0, 1]])
    Ry = np.array([[math.cos(pitch), 0, math.sin(pitch)], [0, 1, 0], [-math.sin(pitch), 0, math.cos(pitch)]])
    Rx = np.array([[1, 0, 0], [0, math.cos(roll), -math.sin(roll)], [0, math.sin(roll), math.cos(roll)]])
    R_gimbal = Rz @ Ry @ Rx
    R_cam_to_body = np.array([[0, 1, 0], [0, 0, 1], [1, 0, 0]]).T
    return R_gimbal @ R_cam_to_body

def calculate_destination_gps(origin_lat, origin_lon, east_m, north_m):
    """
    Calculates the destination GPS coordinates given an origin and offsets in meters.
    """
    if east_m is None or north_m is None: return None, None
    try:
        bearing = math.degrees(math.atan2(east_m, north_m))
        distance_meters = math.hypot(east_m, north_m)
        destination = geodesic(meters=distance_meters).destination(Point(origin_lat, origin_lon), bearing)
        return destination.latitude, destination.longitude
    except Exception as e:
        print(f"ERROR calculating destination GPS: {e}")
        return None, None

def image_point_to_ned(u, v, K, R, abs_depth_map):
    """
    Transforms an image pixel coordinate to a North-East-Down (NED) offset.
    """
    v_idx, u_idx = int(round(v)), int(round(u))
    if not (0 <= v_idx < abs_depth_map.shape[0] and 0 <= u_idx < abs_depth_map.shape[1]):
        print(f"WARNING: Pixel ({u_idx}, {v_idx}) is out of bounds.")
        return None

    distance_to_target = abs_depth_map[v_idx, u_idx]
    K_inv = np.linalg.inv(K)
    ray_cam = K_inv @ np.array([u, v, 1])
    ray_cam_unit = ray_cam / np.linalg.norm(ray_cam)
    point_in_cam_coords = ray_cam_unit * distance_to_target
    ned_offsets = R @ point_in_cam_coords
    return ned_offsets

def gps_to_pixel(target_gps, drone_meta, K, R, base_agl):
    """
    Projects a real-world GPS coordinate onto the 2D image plane.
    """
    geod = Geod(ellps='WGS84')
    drone_lon, drone_lat = drone_meta['longitude'], drone_meta['latitude']
    target_lon, target_lat = target_gps[1], target_gps[0]

    fwd_azi, _, dist = geod.inv(drone_lon, drone_lat, target_lon, target_lat)

    north_offset = dist * math.cos(math.radians(fwd_azi))
    east_offset = dist * math.sin(math.radians(fwd_azi))
    down_offset = base_agl
    ned_offset = np.array([north_offset, east_offset, down_offset])

    point_in_cam_coords = R.T @ ned_offset

    if point_in_cam_coords[2] <= 1e-6:
        return None

    pixel_coords_homogeneous = K @ point_in_cam_coords
    u = pixel_coords_homogeneous[0] / pixel_coords_homogeneous[2]
    v = pixel_coords_homogeneous[1] / pixel_coords_homogeneous[2]

    return int(round(u)), int(round(v))

def calculate_perpendicular_distance(point_gps, line_start_gps, line_end_gps):
    """
    Calculates the shortest distance from a GPS point to a GPS line segment (great-circle path).
    """
    geod = Geod(ellps='WGS84')
    lon_start, lat_start = line_start_gps[1], line_start_gps[0]
    lon_end, lat_end = line_end_gps[1], line_end_gps[0]
    lon_obj, lat_obj = point_gps[1], point_gps[0]

    fwd_azi_transect, _, dist_transect = geod.inv(lon_start, lat_start, lon_end, lat_end)
    if dist_transect < 1e-6:
        return geod.inv(lon_start, lat_start, lon_obj, lat_obj)[2]

    fwd_azi_to_obj, _, dist_start_to_obj = geod.inv(lon_start, lat_start, lon_obj, lat_obj)
    angle_diff_rad = math.radians(fwd_azi_to_obj - fwd_azi_transect)

    perpendicular_dist = dist_start_to_obj * math.sin(angle_diff_rad)
    return perpendicular_dist

def draw_overlays(frame, geolocated_objects_data, transect_pixel_points):
    """
    Draws the projected transect line, bounding boxes, object IDs, and perpendicular distances.
    """
    if all(p is not None for p in transect_pixel_points):
        h, w, _ = frame.shape
        clipped = cv2.clipLine((0, 0, w, h), transect_pixel_points[0], transect_pixel_points[1])
        if clipped[0]:
            p1, p2 = clipped[1], clipped[2]
            cv2.line(frame, p1, p2, (0, 0, 255), 2)

    for data in geolocated_objects_data:
        x1, y1, x2, y2 = data['box']
        object_id = data['object_id']
        distance = data.get('perpendicular_distance', 'N/A')

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        dist_text = f"Dist: {abs(distance):.2f}m" if isinstance(distance, float) else "Dist: N/A"
        text1 = f"ID: {object_id}"

        cv2.putText(frame, text1, (x1, y1 - 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        cv2.putText(frame, dist_text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    return frame

def save_transect_data_to_csv(filename, image_filename_stem, geolocated_objects, drone_gps):
    """
    Saves the detailed transect sampling data to a CSV file, formatted for R.
    """
    filepath = os.path.join(OUTPUT_DIR, filename)
    header = [
        'image_id', 'drone_latitude', 'drone_longitude', 'object_id', 'class_id',
        'object_latitude', 'object_longitude', 'perpendicular_distance_m'
    ]
    with open(filepath, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(header)
        for obj in geolocated_objects:
            lat, lon = obj['gps']
            dist = obj.get('perpendicular_distance')
            writer.writerow([
                image_filename_stem,
                f"{drone_gps[0]:.8f}", f"{drone_gps[1]:.8f}",
                obj['object_id'], obj['class_id'],
                f"{lat:.8f}", f"{lon:.8f}",
                f"{dist:.4f}" if isinstance(dist, float) else "N/A"
            ])
    print(f"✅ Line-transect data for R saved to: {filepath}")
    return filepath # Return path for downloading

# --- MAIN EXECUTION BLOCK ---
def main_image_processing_pipeline(image_path, object_to_detect):
    print(f"\n--- 🚀 Starting Pipeline for {Path(image_path).name} 🚀 ---")

    try:
        print("INFO: Loading AI models...")
        depth_processor = AutoImageProcessor.from_pretrained(DEPTH_MODEL_NAME, use_fast=True)
        depth_model = AutoModelForDepthEstimation.from_pretrained(DEPTH_MODEL_NAME).to(DEVICE)
        print("✅ Models loaded successfully.")

        image_filename_stem = Path(image_path).stem

        meta = extract_exif_data(image_path)
        if not meta: return

        frame = cv2.imread(image_path)
        if frame is None:
            print(f"❌ ERROR: Cannot open image file {image_path}")
            return
        frame_height, frame_width, _ = frame.shape

        detected_objects = detect_objects_yolo(object_to_detect, image_path)

        print("\n--- Starting Georeferencing Calculations ---")
        ground_elevation = get_dem_elevation_from_api(meta['latitude'], meta['longitude'])
        base_agl = (meta['abs_alt'] - ground_elevation) if ground_elevation is not None else meta['rel_alt']
        print(f"INFO: Ground elevation: {ground_elevation or 'N/A'}. Camera AGL: {base_agl:.2f}m.")

        relative_depth_map = get_depth_map(frame, depth_model, depth_processor)
        center_h, center_w = frame_height // 2, frame_width // 2
        central_depth_region = relative_depth_map[center_h-10:center_h+10, center_w-10:center_w+10]
        center_pixel_depth = np.mean(central_depth_region)
        scale_factor = base_agl / center_pixel_depth if center_pixel_depth > 1e-6 else 1.0
        absolute_depth_map = relative_depth_map * scale_factor

        K = get_camera_intrinsics(meta['focal_len'], SENSOR_WIDTH_MM, SENSOR_HEIGHT_MM, frame_width, frame_height)
        R = get_rotation_matrix(meta['gb_pitch'], meta['gb_yaw'], meta['gb_roll'])

        # Define the Transect Line based on Drone's Ground Track
        drone_point = Point(meta['latitude'], meta['longitude'])
        drone_bearing = meta['gb_yaw']
        transect_start_gps = geodesic(meters=-50).destination(drone_point, drone_bearing)
        transect_end_gps = geodesic(meters=50).destination(drone_point, drone_bearing)
        print(f"✅ Transect line defined by drone's ground track (Bearing: {drone_bearing:.2f}°).")

        # Project transect line points onto image for drawing
        pixel_nadir = gps_to_pixel((meta['latitude'], meta['longitude']), meta, K, R, base_agl)
        pixel_forward = gps_to_pixel((transect_end_gps.latitude, transect_end_gps.longitude), meta, K, R, base_agl)
        transect_pixel_points = (None, None)
        if pixel_nadir and pixel_forward:
            dx = pixel_forward[0] - pixel_nadir[0]
            dy = pixel_forward[1] - pixel_nadir[1]
            p1 = (pixel_nadir[0] - dx * 100, pixel_nadir[1] - dy * 100)
            p2 = (pixel_nadir[0] + dx * 100, pixel_nadir[1] + dy * 100)
            transect_pixel_points = (p1, p2)
        else:
            print("⚠️  WARNING: Could not reliably project the drone's ground track onto the image.")

        geolocated_objects = []
        if detected_objects:
            for obj_data in detected_objects:
                ned_offset = image_point_to_ned(obj_data['center_u'], obj_data['center_v'], K, R, absolute_depth_map)
                if ned_offset is not None:
                    lat, lon = calculate_destination_gps(meta['latitude'], meta['longitude'], ned_offset[1], ned_offset[0])
                    if lat is not None:
                        obj_data['gps'] = (lat, lon)
                        dist = calculate_perpendicular_distance(obj_data['gps'],
                                                                (transect_start_gps.latitude, transect_start_gps.longitude),
                                                                (transect_end_gps.latitude, transect_end_gps.longitude))
                        obj_data['perpendicular_distance'] = dist
                        geolocated_objects.append(obj_data)
                        print(f"  > Object {obj_data['object_id']} geolocated. Perpendicular distance: {dist:.2f}m")
        else:
            print("INFO: No objects were detected to geolocate.")

        # --- Generate and Download Outputs ---
        annotated_frame = draw_overlays(frame.copy(), geolocated_objects, transect_pixel_points)
        output_image_path = os.path.join(OUTPUT_DIR, f"{image_filename_stem}_transect_annotated.jpg")
        cv2.imwrite(output_image_path, annotated_frame)
        print(f"\n✅ Annotated image saved to: {output_image_path}")
        files.download(output_image_path) # Auto-download the image

        csv_filename = f"{image_filename_stem}_drone_transect_data.csv"
        csv_filepath = save_transect_data_to_csv(
            csv_filename, image_filename_stem, geolocated_objects, (meta['latitude'], meta['longitude'])
        )
        files.download(csv_filepath) # Auto-download the CSV

    except Exception as e:
        print(f"\n❌ An unexpected error occurred: {e}")
        traceback.print_exc()

## 🧠 2. Downloading Pre-trained Models and Scripts

Now, we need to download the "brains" of our operation. This section downloads two critical files from Google Drive:

1.  **Inference Script (`single_image_inference_...py`)**: A custom Python script that runs the object detection model.
2.  **Model Weights (`checkpoint_fsc147_best.pth`)**: This is the pre-trained model file containing all the learned information for detecting objects. It's over 1GB in size, so the download may take a moment.

We also apply a quick patch to the script to ensure it can find the text model (`bert-base-uncased`) correctly. The cell will verify that both files have been downloaded successfully.

In [13]:
# Install the gdown library, which is excellent for downloading from Google Drive
!pip install -q gdown
import os
import requests

# --- USER ACTION REQUIRED: Paste your Google Drive File IDs below ---
PY_SCRIPT_FILE_ID = "10WSG3TPB1jC_08b7QOhFPniRas0IHMmt"
PTH_MODEL_FILE_ID = "1sFfo_T9wyvP7fXyLXxe7LnY4NSpkzhWj"
# --------------------------------------------------------------------

# --- Define file paths ---
SCRIPT_DESTINATION = 'single_image_inference_for_DirectGeoReferencingUSe.py'
MODEL_DESTINATION_DIR = 'checkpoints'
MODEL_DESTINATION_PATH = os.path.join(MODEL_DESTINATION_DIR, 'checkpoint_fsc147_best.pth')

# Create the checkpoints directory if it doesn't exist
os.makedirs(MODEL_DESTINATION_DIR, exist_ok=True)

# --- Download the files using gdown ---
print("Starting download process...")
!gdown --id {PY_SCRIPT_FILE_ID} -O {SCRIPT_DESTINATION}
!gdown --id {PTH_MODEL_FILE_ID} -O {MODEL_DESTINATION_PATH}

# --- Verification ---
print("\nVerifying downloads...")
if os.path.exists(SCRIPT_DESTINATION) and os.path.getsize(SCRIPT_DESTINATION) > 0:
    # Check if the file is a python script and not an HTML page
    with open(SCRIPT_DESTINATION, 'r') as f:
        first_line = f.readline()
    if '<!DOCTYPE html>' in first_line:
        print(f"❌ ERROR: Download failed for '{SCRIPT_DESTINATION}'. It is still an HTML file. Please double-check your File ID and sharing permissions.")
    else:
        print(f"✅ Success: '{SCRIPT_DESTINATION}' is in the correct location.")
else:
    print(f"❌ ERROR: Failed to download '{SCRIPT_DESTINATION}'.")

if os.path.exists(MODEL_DESTINATION_PATH) and os.path.getsize(MODEL_DESTINATION_PATH) > 0:
    print(f"✅ Success: '{os.path.basename(MODEL_DESTINATION_PATH)}' is in the '{MODEL_DESTINATION_DIR}' directory.")
else:
    print(f"❌ ERROR: Failed to download model checkpoint. Please double-check your File ID and sharing permissions.")

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/MultiScaleDeformableAttention-1.0-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
Starting download process...
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=10WSG3TPB1jC_08b7QOhFPniRas0IHMmt
From (redirected): https://drive.google.com/uc?id=10WSG3TPB1jC_08b7QOhFPniRas0IHMmt&confirm=t&uuid=79352238-7c62-4440-a89e-9607476afc40
To: /content/CountGD/single_image_inference_for_DirectGeoReferencingUSe.py
100% 7.18k/7.18k [00:00<00:00, 26.4MB/s]
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option

In [14]:
# --- 2. CRITICAL FIX: Correct the hardcoded BERT model path in the script ---
print("\nPatching the inference script to fix the model path...")
!sed -i 's|checkpoints/bert-base-uncased|bert-base-uncased|g' {SCRIPT_DESTINATION}
print("✅ Script patched successfully.")


Patching the inference script to fix the model path...
✅ Script patched successfully.


## 🚀 3. Run the Analysis!

This is the final step where you get to process your own data.

### Instructions:

1.  **Define Your Target Object**: In the first line of the code cell below, change the text `"Elephant"` to whatever object you want to detect (e.g., `"Car"`, `"Boat"`, `"Cow"`). **Keep it singular and capitalized.**
2.  **Run the Cell**: Execute the code cell.
3.  **Upload Your Image(s)**: A file upload widget will appear. Click "Choose Files" and select one or more drone images from your computer. The images must contain EXIF metadata (GPS location, altitude, gimbal angles).

The pipeline will then run automatically for each image you've uploaded.

In [15]:
# --- USER INPUTS ---
OBJECT_TO_DETECT = "Elephant"  # <-- CHANGE THIS to "Deer", "Car", etc.
# -------------------

print(f"Starting analysis to detect: {OBJECT_TO_DETECT}")
print("Please upload the drone image file(s) you want to process.")

# Use Colab's file uploader
uploaded_files = files.upload()

if not uploaded_files:
    print("\nNo files were uploaded. Please run the cell again to upload images.")
else:
    for filename, content in uploaded_files.items():
        # Write the uploaded file to the local Colab environment
        with open(filename, 'wb') as f:
            f.write(content)

        # Check if the file exists before processing
        if os.path.exists(filename):
            main_image_processing_pipeline(filename, OBJECT_TO_DETECT)
        else:
            print(f"❌ ERROR: Failed to save uploaded file '{filename}'.")

    print("\n\n--- 🎉 All processing complete. Check your downloads folder for the results. ---")

Starting analysis to detect: Elephant
Please upload the drone image file(s) you want to process.


Saving DJI_0396.JPG to DJI_0396.JPG

--- 🚀 Starting Pipeline for DJI_0396.JPG 🚀 ---
INFO: Loading AI models...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

<IPython.core.display.Javascript object>

✅ Line-transect data for R saved to: Image_Processing_Output/DJI_0396_drone_transect_data.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



--- 🎉 All processing complete. Check your downloads folder for the results. ---


## 📊 4. Understanding the Outputs

After the processing is complete, two files will be automatically downloaded for each input image:

### 1. Annotated Image (`*_transect_annotated.jpg`)
This is a visual representation of the analysis. It shows:
* **Green Bounding Boxes**: The objects detected by the model.
* **Object ID**: A unique number for each detected object in the image.
* **Perpendicular Distance**: The calculated shortest distance (in meters) from the object to the drone's flight path.
* **Red Line**: The projected transect line (the drone's ground track).


### 2. CSV Data File (`*_drone_transect_data.csv`)
This file contains the structured data ready for statistical analysis, especially for distance sampling in programs like **R**. It includes the following columns:

| Column Name                | Description                                                              |
| -------------------------- | ------------------------------------------------------------------------ |
| `image_id`                 | The name of the source image file.                                       |
| `drone_latitude`           | The latitude of the drone when the photo was taken.                      |
| `drone_longitude`          | The longitude of the drone when the photo was taken.                     |
| `object_id`                | The unique identifier for the detected object.                           |
| `class_id`                 | The class label for the object (will be 0 for your specified object).    |
| `object_latitude`          | The calculated latitude of the detected object.                          |
| `object_longitude`         | The calculated longitude of the detected object.                         |
| `perpendicular_distance_m` | The key measurement: the perpendicular distance in meters from the transect. |

---
**🎉 You have successfully completed the analysis!**